In [2]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import os
import matplotlib.pyplot as plt
import numpy as np
import suite2p
import mbo_utilities as mbo
import fastplotlib as fpl
from copy import deepcopy
import lbm_suite2p_python as lsp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
files = mbo.get_files("~/lbm_data/demo/assembled", 'tif')
metadata = mbo.get_metadata(files[0])
ops = lsp.default_ops(metadata=metadata)
ops["keep_movie_raw"] = True
ops["reg_tif"] = True

In [4]:
files[0]

'/home/flynn/lbm_data/demo/assembled/plane_01_demo.tiff'

In [5]:
files[0], files[1]

('/home/flynn/lbm_data/demo/assembled/plane_01_demo.tiff',
 '/home/flynn/lbm_data/demo/assembled/plane_07_demo.tiff')

In [65]:
import re
import traceback
from pathlib import Path

import numpy as np
import mbo_utilities as mbo
import suite2p
from suite2p.run_s2p import run_plane as s2p_run_plane
from tifffile import imread
from scipy.ndimage import uniform_filter1d

from lbm_suite2p_python.zplane import (
    load_planar_results,
    plot_traces,
    plot_noise_distribution,
    plot_projection,
)

try:
    from rastermap import Rastermap
    HAS_RASTERMAP = True
except ImportError:
    HAS_RASTERMAP = False


def run_plane(ops, input_tiff, save_path,
              overwrite=False, replot=False, dryrun=False, **kwargs):
    """
    Process a single‐plane TIFF with Suite2p into save_path/planeN/,
    where N = int(digits after 'plane_') - 1.  No extra 'plane0' folder.

    Parameters
    ----------
    ops : dict
        Your base Suite2p ops.
    input_tiff : str or Path
        Path to a single‐plane TIFF named like '...plane_07_....tiff'.
    save_path : str or Path
        Directory in which to create 'plane6/', 'plane7/', etc.
    overwrite : bool
        If True, rerun even if ops.npy/stat.npy already exist.
    replot : bool
        If True, regenerate the registration/segmentation/traces plots.
    dryrun : bool
        If True, just print what would happen.
    """
    input_tiff = Path(input_tiff)
    if not input_tiff.is_file():
        raise FileNotFoundError(f"{input_tiff} not found.")

    # figure out which plane folder to use
    m = re.search(r"plane_(\d+)", input_tiff.stem)
    plane_idx = int(m.group(1)) - 1 if m else 0
    folder_name = f"plane{plane_idx}"

    base_dir = Path(save_path).expanduser().resolve()
    if not base_dir.is_dir():
        raise NotADirectoryError(f"{base_dir} is not a directory.")
    plane_dir = base_dir / folder_name
    plane_dir.mkdir(exist_ok=True)

    if dryrun:
        print(f"[dryrun] would write data.bin + Suite2p outputs into {plane_dir}")
        return

    # --- 1) TIFF → data.bin (no nested plane0 folder) ---
    stack = imread(input_tiff)                        # shape = (nframes, Ly, Lx)
    nframes, Ly, Lx = stack.shape
    bin_path = plane_dir / "data.bin"
    with open(bin_path, "wb") as f:
        f.write(stack.tobytes())

    # --- 2) Prepare ops for low-level Suite2p run_plane ---
    #  a) if ops was None pull defaults
    if ops is None:
        ops = suite2p.default_ops()
    #  b) fill in metadata
    metadata = mbo.get_metadata(input_tiff)
    ops = mbo.params_from_metadata(metadata, ops)
    #  c) override only the bits run_plane needs:
    ops.update({
        "reg_file": str(bin_path),
        "nframes": int(nframes),
        "Ly":      int(Ly),
        "Lx":      int(Lx),
    })
    # save an ops.npy next
    ops_path = plane_dir / "ops.npy"
    np.save(str(ops_path), ops)

    # --- 3) Call Suite2p’s low-level run_plane ---
    output_ops = s2p_run_plane(ops, ops_path=str(ops_path))

    return output_ops


In [67]:
output_ops = run_plane(ops, files[0], "~/lbm_data/demo")

Ops provided. Setting pipeline to suite2p
NOTE: not registered / registration forced with ops['do_registration']>1
      (no previous offsets to delete)
NOTE: applying default /home/flynn/.suite2p/classifiers/classifier_user.npy
----------- REGISTRATION
Reference frame, 7.67 sec.
Registered 500/1437 in 7.74s
Registered 1000/1437 in 15.13s
Registered 1437/1437 in 21.58s
----------- Total 34.24 sec
----------- ROI DETECTION
Binning movie in chunks of length 17
Binned movie of size [84,438,438] created in 0.34 sec.
NOTE: estimated spatial scale ~6 pixels, time epochs 1.00, threshold 5.00 
0 ROIs, score=23.39
Detected 66 ROIs, 0.75 sec
After removing overlaps, 63 ROIs remain
----------- Total 1.17 sec.
----------- EXTRACTION
Masks created, 0.12 sec.
Extracted fluorescence from 63 ROIs in 1437 frames, 0.34 sec.
----------- Total 0.48 sec.
----------- CLASSIFICATION
['compact', 'npix_norm', 'skew']
----------- SPIKE DECONVOLUTION
----------- Total 0.00 sec.


In [71]:
ops["do_registration"] = False

In [72]:
output_ops = lsp.run_plane(ops, files[1], "~/lbm_data/demo")

{'data_path': ['/home/flynn/lbm_data/demo/assembled'], 'tiff_list': ['plane_07_demo.tiff'], 'save_path0': '/home/flynn/lbm_data/demo', 'save_folder': 'plane6'}
tif
** Found 1 tifs - converting to binary **
time 1.24 sec. Wrote 1437 frames per binary for 1 planes
>>>>>>>>>>>>>>>>>>>>> PLANE 0 <<<<<<<<<<<<<<<<<<<<<<
NOTE: not running registration, ops['do_registration']=0
binary path: /home/flynn/lbm_data/demo/plane6/plane0/data.bin
NOTE: applying default /home/flynn/.suite2p/classifiers/classifier_user.npy
----------- ROI DETECTION
Binning movie in chunks of length 17
Binned movie of size [84,444,444] created in 0.50 sec.
NOTE: FORCED spatial scale ~6 pixels, time epochs 1.00, threshold 5.00 
Detected 0 ROIs, 0.42 sec


ValueError: no ROIs were found -- check registered binary and maybe change spatial scale

In [24]:
from suite2p.io import tiff_to_binary